In [1]:
# 03 RFM Customer Segmentation.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

pd.set_option('display.max_columns', None)

In [3]:
password = getpass("retail_user password: ")

url = URL.create(
    drivername="mysql+pymysql",
    username="retail_user",
    password=password,
    host="localhost",
    port=3306,
    database="retail_analysis"
)
engine = create_engine(url)

with engine.connect() as conn:
    print("Connected successfully")

retail_user password:  ········


Connected successfully


In [5]:
customers = pd.read_sql("SELECT * FROM v_customer_summary", engine)
print(customers.shape)
customers.head()

(4334, 7)


,customer_id,country,orders,total_spent,first_order,last_order,avg_order_value
0,12346,United Kingdom,1,77183.60,2011-01-18,2011-01-18,77183.60
1,12347,Iceland,7,4310.00,2010-12-07,2011-12-07,615.71
2,12348,Finland,4,1437.24,2010-12-16,2011-09-25,359.31
3,12349,Italy,1,1457.55,2011-11-21,2011-11-21,1457.55
4,12350,Norway,1,294.40,2011-02-02,2011-02-02,294.40


In [6]:
customers['first_order'] = pd.to_datetime(customers['first_order'])
customers['last_order']  = pd.to_datetime(customers['last_order'])
customers['total_spent'] = customers['total_spent'].astype(float)
customers.dtypes

customer_id                int64
country                      str
orders                     int64
total_spent              float64
first_order        datetime64[s]
last_order         datetime64[s]
avg_order_value          float64
dtype: object

In [7]:
snapshot_date = customers['last_order'].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

Snapshot date: 2011-12-10 00:00:00


In [8]:
rfm = pd.DataFrame({
    'customer_id': customers['customer_id'],
    'country':     customers['country'],
    'recency':     (snapshot_date - customers['last_order']).dt.days,
    'frequency':   customers['orders'],
    'monetary':    customers['total_spent']
})
rfm.head()

,customer_id,country,recency,frequency,monetary
0,12346,United Kingdom,326,1,77183.60
1,12347,Iceland,3,7,4310.00
2,12348,Finland,76,4,1437.24
3,12349,Italy,19,1,1457.55
4,12350,Norway,311,1,294.40


In [9]:
print("Rows:", len(rfm))
print("Unique customers:", rfm['customer_id'].is_unique)
print("Total monetary:", round(rfm['monetary'].sum(), 2))
rfm[['recency', 'frequency', 'monetary']].describe()

Rows: 4334
Unique customers: True
Total monetary: 8737227.64


,recency,frequency,monetary
count,4334.000000,4334.000000,4334.000000
mean,93.226581,4.245962,2015.973152
std,100.175327,7.634989,8903.673825
min,1.000000,1.000000,3.750000
25%,18.000000,1.000000,304.240000
50%,51.000000,2.000000,662.565000
75%,143.000000,5.000000,1631.622500
max,374.000000,206.000000,279138.020000


In [10]:
rfm['frequency'].value_counts().sort_index().head(12)

frequency
1     1505
2      831
3      502
4      395
5      236
6      173
7      139
8       98
9       69
10      54
11      54
12      43
Name: count, dtype: int64

In [11]:
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

In [12]:
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

In [13]:
rfm['f_score'] = pd.cut(rfm['frequency'],
                        bins=[0, 1, 2, 4, 8, np.inf],
                        labels=[1, 2, 3, 4, 5]).astype(int)

In [14]:
for col in ['r_score', 'f_score', 'm_score']:
    print(col)
    print(rfm[col].value_counts().sort_index())
    print()

r_score
r_score
1    863
2    861
3    865
4    840
5    905
Name: count, dtype: int64

f_score
f_score
1    1505
2     831
3     897
4     646
5     455
Name: count, dtype: int64

m_score
m_score
1    867
2    867
3    866
4    867
5    867
Name: count, dtype: int64



In [15]:
rfm.groupby('r_score')['recency'].agg(['min', 'max', 'count'])

,min,max,count
r_score,,,
1,181,374,863
2,73,180,861
3,34,72,865
4,16,33,840
5,1,15,905


In [16]:
conditions = [
    (rfm['r_score'] >= 4) & (rfm['f_score'] >= 4) & (rfm['m_score'] >= 4),  # Champions
    (rfm['r_score'] >= 3) & (rfm['f_score'] >= 3),                          # Loyal Customers
    (rfm['r_score'] >= 4) & (rfm['f_score'] == 1),                          # New Customers
    (rfm['r_score'] >= 4) & (rfm['f_score'] == 2),                          # Potential Loyalists
    (rfm['r_score'] <= 2) & (rfm['f_score'] >= 3),                          # At Risk
    (rfm['r_score'] <= 2) & (rfm['f_score'] <= 2),                          # Lost
]

labels = ['Champions', 'Loyal Customers', 'New Customers',
          'Potential Loyalists', 'At Risk', 'Lost']

rfm['segment'] = np.select(conditions, labels, default='Needs Attention')

In [17]:
print(rfm['segment'].isnull().sum())          # should be 0
print(rfm['segment'].value_counts())
print(rfm['segment'].value_counts().sum())    # should be 4334

0
segment
Lost                   1340
Loyal Customers         834
Champions               780
Needs Attention         478
At Risk                 384
Potential Loyalists     280
New Customers           238
Name: count, dtype: int64
4334


In [18]:
summary = (rfm.groupby('segment')
           .agg(customers=('customer_id', 'count'),
                revenue=('monetary', 'sum'),
                avg_recency_days=('recency', 'mean'),
                avg_orders=('frequency', 'mean'),
                avg_spend=('monetary', 'mean'))
           .reset_index())

summary['pct_customers'] = 100 * summary['customers'] / summary['customers'].sum()
summary['pct_revenue']   = 100 * summary['revenue'] / summary['revenue'].sum()

summary = summary.round(2).sort_values('revenue', ascending=False).reset_index(drop=True)
summary

,segment,customers,revenue,avg_recency_days,avg_orders,avg_spend,pct_customers,pct_revenue
0,Champions,780,5354019.59,12.47,12.67,6864.13,18.00,61.28
1,Loyal Customers,834,1397676.31,32.17,4.42,1675.87,19.24,16.00
2,Lost,1340,678038.82,209.44,1.28,506.00,30.92,7.76
3,At Risk,384,638035.26,136.03,4.35,1661.55,8.86,7.30
4,Potential Loyalists,280,344993.80,16.45,2.00,1232.12,6.46,3.95
5,Needs Attention,478,248817.02,53.10,1.37,520.54,11.03,2.85
6,New Customers,238,75646.84,19.40,1.00,317.84,5.49,0.87


In [19]:
print(summary['customers'].sum(), round(summary['revenue'].sum(), 2))

4334 8737227.64


In [20]:
rfm.to_csv('../data/cleaned/rfm_segments.csv', index=False)
summary.to_csv('../data/cleaned/rfm_segment_summary.csv', index=False)
